# 2. Feature Engineering
Detailed feature preparation notebook aligned with the production pipeline.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config.core import config
from src.pipeline import pipe
from src.processing.data_manager import load_dataset
from src.processing.features import Mapper

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent


In [ ]:
data = load_dataset(config.app_config.training_data_file)
print('Shape after dropping configured columns:', data.shape)
display(data.head(3))


In [ ]:
x = data.drop(columns=[config.ml_config.target]).copy()
y = data[config.ml_config.target].astype(int).copy()

x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=config.ml_config.test_size,
    random_state=config.ml_config.random_state,
    stratify=y,
)

print('x_train:', x_train.shape, '| x_valid:', x_valid.shape)


In [ ]:
x_manual = x_train.copy()

for col in config.ml_config.categorical_missing_with_constant:
    x_manual[col] = x_manual[col].fillna('Missing')

for col in config.ml_config.categorical_missing_with_frequent:
    mode_val = x_manual[col].mode(dropna=True)[0]
    x_manual[col] = x_manual[col].fillna(mode_val)

missing_after_impute = x_manual.isna().sum().sum()
print('Total missing values after manual categorical imputation:', missing_after_impute)


In [ ]:
experience_mapper = Mapper(
    variables=config.ml_config.experience_features,
    mappings=config.ml_config.experience_map,
    default_value=0,
)
last_job_mapper = Mapper(
    variables=config.ml_config.last_new_job_features,
    mappings=config.ml_config.last_new_job_map,
    default_value=0,
)
company_size_mapper = Mapper(
    variables=config.ml_config.company_size_features,
    mappings=config.ml_config.company_size_map,
    default_value=0,
)

x_mapped = experience_mapper.transform(x_manual)
x_mapped = last_job_mapper.transform(x_mapped)
x_mapped = company_size_mapper.transform(x_mapped)

display(x_mapped[config.ml_config.experience_features + config.ml_config.last_new_job_features + config.ml_config.company_size_features].head())


In [ ]:
prep_pipe = pipe[:-1]
x_train_transformed = prep_pipe.fit_transform(x_train, y_train)
x_valid_transformed = prep_pipe.transform(x_valid)

print('Transformed train shape:', x_train_transformed.shape)
print('Transformed valid shape:', x_valid_transformed.shape)
print('NaN count in transformed train:', np.isnan(x_train_transformed).sum())
print('NaN count in transformed valid:', np.isnan(x_valid_transformed).sum())


In [ ]:
full_pipe = pipe
full_pipe.fit(x_train, y_train)
train_score = full_pipe.score(x_train, y_train)
valid_score = full_pipe.score(x_valid, y_valid)

print(f'Train accuracy: {train_score:.4f}')
print(f'Valid accuracy: {valid_score:.4f}')


This notebook demonstrates feature processing both manually (for explainability) and through the final sklearn pipeline (for production consistency).